# Evaluate FT BERT

In this notebook, we fine tune BERT on a document classification task, compare the performance against the pretrained BERT (also with a trained linear projection from the cls token to the 11 classes on the same hyperparameters but frozen BERT parameters)

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import BertModel, BertTokenizer
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from lxml import etree

Read in data

In [2]:
import xml.etree.ElementTree as ET

def parse_xml_file(filepath):
    tree = ET.parse(filepath)
    root = tree.getroot()

    current_part = ""
    current_section = ""
    results = []

    for norm in root.findall(".//norm"):
        # Check for Part or Section in this norm
        gliederungseinheit = norm.find(".//gliederungseinheit")
        if gliederungseinheit is not None:
            kennzahl = gliederungseinheit.findtext("gliederungskennzahl")
            bez = gliederungseinheit.findtext("gliederungsbez")
            titel = gliederungseinheit.findtext("gliederungstitel")
            if kennzahl and len(kennzahl) == 3:
                current_part = f"{bez} - {titel}"
            elif kennzahl and len(kennzahl) == 6:
                current_section = f"{bez} - {titel}"

        # Subsection title
        subsection = norm.findtext(".//titel[@format='XML']")

        # Content paragraph
        for p in norm.findall(".//textdaten/text/Content/P"):
            content_text = ''.join(p.itertext()).strip()

            if content_text:
                results.append({
                    "metadata": {
                        "subtitle": current_part,
                        "section": current_section,
                        "subsection": subsection
                    },
                    "content": content_text,
                })

    return results


class SubtitleDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

Define Model

In [3]:
class BertSubtitleClassifier(nn.Module):
    def __init__(self, num_labels, freeze_bert=False):
            super().__init__()
            self.bert = BertModel.from_pretrained("bert-base-german-cased")
            if freeze_bert:
                for param in self.bert.parameters():
                    param.requires_grad = False  # Disable autograd so we can compare the finetuned to pretrained
            self.dropout = nn.Dropout(0.3)
            self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

Define Training and Eval Functions

In [4]:
def evaluate(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    print(f"  ➤ Accuracy: {acc:.4f} | Macro-F1: {f1_macro:.4f}")
    return f1_macro  # Return F1 for cross-validation comparison


def train_with_early_stopping(model, train_loader, val_loader, optimizer, criterion, device, epochs=5, patience=2):
    best_f1 = 0.0
    best_model_state = None
    epochs_no_improve = 0

    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")

        # Training step
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc="Training"):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f" Avg train loss: {avg_loss:.4f}")

        # Validation step
        f1_macro = evaluate(model, val_loader, device)

        # Early stopping logic
        if f1_macro > best_f1:
            best_f1 = f1_macro
            best_model_state = model.state_dict()
            epochs_no_improve = 0
            print(f"  🎉 New best Macro-F1: {best_f1:.4f}")
        else:
            epochs_no_improve += 1
            print(f"  No improvement for {epochs_no_improve} epochs.")

        if epochs_no_improve >= patience:
            print(f"Stopping early after {epoch+1} epochs.")
            break

    # Load best weights before returning
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model


def train_final_model(parsed_data, num_epochs, batch_size, lr_bert, lr_class, patience=2, device=None, freeze_bert=False, save_dir="bert-subtitle-embedder"):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Extract data
    texts = [entry["content"] for entry in parsed_data]
    subtitles = [entry["metadata"]["subtitle"] for entry in parsed_data]

    # Encode labels
    label_encoder = LabelEncoder()
    labels = label_encoder.fit_transform(subtitles)

    # Train-val split (90/10) for early stopping
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        texts, labels, test_size=0.1, stratify=labels, random_state=42
    )

    tokenizer = BertTokenizer.from_pretrained("bert-base-german-cased")

    # Dataset and loaders
    train_dataset = SubtitleDataset(train_texts, train_labels, tokenizer)
    val_dataset = SubtitleDataset(val_texts, val_labels, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    # Init model
    model = BertSubtitleClassifier(num_labels=len(label_encoder.classes_)).to(device)
    optimizer = AdamW([
            {'params': model.bert.parameters(), 'lr': lr_bert},
            {'params': model.classifier.parameters(), 'lr': lr_class}
        ])
    criterion = nn.CrossEntropyLoss()

    # Train with early stopping
    model = train_with_early_stopping(
        model, train_loader, val_loader, optimizer, criterion, device,
        epochs=num_epochs, patience=patience
    )

    print("\n\nFinal Evaluation:\n")
    evaluate(model, val_loader, device)

    # Save only fine-tuned BERT encoder if not frozen
    if not freeze_bert:
        model.bert.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"Fine-tuned model saved to {save_dir}")
    else:
        print("Frozen BERT model trained and evaluated (not saved).")

    return model

Call train and use best hyperparameters

In [5]:
# Hyperparameters
lr_bert = 3e-05
lr_class = 0.001
epochs = 4
batch_size = 16
patience = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_dir = "/storage/Tax_Law_RAG/german-tax-law/output/EStG_Subtitle_Classification_BERT"

# Data
xml_file = "/storage/Tax_Law_RAG/german-tax-law/EStG.xml"
parsed_data = parse_xml_file(xml_file)

Evaluate BERT and FT BERT on F1 and Accuracy - Note stdout

def train_final_model(parsed_data, num_epochs, batch_size, lr, patience=2, device=None, freeze_bert=False, save_dir="bert-subtitle-embedder"):

In [8]:
# Train Fine Tuned BERT
train_final_model(parsed_data=parsed_data, num_epochs=epochs, batch_size=batch_size, lr_bert=lr_bert, lr_class=lr_class, patience=patience, device=device, freeze_bert=False, save_dir=save_dir)

Epoch 1/4


Training: 100%|██████████| 59/59 [00:29<00:00,  2.03it/s]


 Avg train loss: 1.6893


Evaluating: 100%|██████████| 7/7 [00:01<00:00,  5.77it/s]


  ➤ Accuracy: 0.5619 | Macro-F1: 0.2047
  🎉 New best Macro-F1: 0.2047
Epoch 2/4


...

Evaluating: 100%|██████████| 7/7 [00:01<00:00,  5.69it/s]


  ➤ Accuracy: 0.7429 | Macro-F1: 0.7296
  🎉 New best Macro-F1: 0.7296


Final Evaluation:



Evaluating: 100%|██████████| 7/7 [00:01<00:00,  5.69it/s]


  ➤ Accuracy: 0.7429 | Macro-F1: 0.7296
Fine-tuned model saved to /storage/Tax_Law_RAG/german-tax-law/output/EStG_Subtitle_Classification_BERT


BertSubtitleClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [9]:
# Train the classifier (linear projection) on frozen BERT
train_final_model(parsed_data=parsed_data, num_epochs=epochs, batch_size=batch_size, lr_bert=lr_bert, lr_class=lr_class, patience=patience, device=device, freeze_bert=True, save_dir=save_dir)

Epoch 1/4


Training: 100%|██████████| 59/59 [00:29<00:00,  2.00it/s]


 Avg train loss: 1.7473


Evaluating: 100%|██████████| 7/7 [00:01<00:00,  5.70it/s]


  ➤ Accuracy: 0.5619 | Macro-F1: 0.2933
  🎉 New best Macro-F1: 0.2933
Epoch 2/4


...

Evaluating: 100%|██████████| 7/7 [00:01<00:00,  5.72it/s]


  ➤ Accuracy: 0.6667 | Macro-F1: 0.5798
  No improvement for 1 epochs.


Final Evaluation:



Evaluating: 100%|██████████| 7/7 [00:01<00:00,  5.69it/s]

  ➤ Accuracy: 0.6667 | Macro-F1: 0.5798
Frozen BERT model trained and evaluated (not saved).


BertSubtitleClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el